In [ ]:
using CSV 
using DataFrames

using Turing
using KernelFunctions
using LinearAlgebra

using ReverseDiff
using Plots

using MCMCChains
using Serialization 

using BayesSoundSource

In [ ]:
microphone_coords = [
    [0.0, 0.0, 0.0],
    [-8.0, 0.0, 0.89],
    [0.0, 8.0, 1.65],
    [8.0, 0.0, 0.92],
    [0.0, -8.0, 1.64]
]


folder  = "/home/-/Documents/Field-data/bat_oct25/extracts/20251021_173336"
data = CSV.read(joinpath(folder, "GCC_data.csv"), DataFrame)


tdoa_set = collect(eachrow(Array(data[!, ["tdoa_$(i)_$j" for i ∈ 1:5 for j ∈ i+1:5]])))
toa_set = collect(eachrow(Array(data[!, ["toa_$(i)" for i ∈ 1:5]])))
outlier_resp = collect(eachrow(Array(data[!, ["outlier_resp_$(i)_$j" for i ∈ 1:5 for j ∈ i+1:5]])))

outlier_filt = map(o -> all(o .< 0.1) , outlier_resp)

tdoa_set = tdoa_set[outlier_filt]
toa_set = toa_set[outlier_filt]
data

In [ ]:
toa_ls = map(tdoa_set[outlier_filt]) do tdoa 
    y, _ = tdoa_mle(microphone_coords, tdoa, speed_of_sound)
    return y
end 

In [ ]:
receiver_priors = MvNormal.(microphone_coords, 0.1)
traj_prior = GPTrajPrior(3)
# traj_prior = FlatTrajPrior()

model = toa_tdoa_model(tdoa_set, toa_set, receiver_priors, traj_prior)

chn = sample(model, 
    NUTS(; adtype=AutoReverseDiff(compile=true)), 
    300; 
    initial_params=InitFromPrior(),
    chain_type=MCMCChains.Chains)

serialize("chn_joint.bin", chn)
describe(chn)
